In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


# Selective Attention (Stroop Analogue) Benchmark
**Cognitive Science**: Stroop (1935)
Tests ability to focus on relevant info while ignoring distractors

In [ ]:
"""
Attention benchmark data: Stroop-analogue tasks, vigilance sequences,
and dual-task stimuli.
"""
import random
import hashlib
# ─── Stroop Analogue ────────────────────────────────────────────────
# Instead of color words in wrong colors, we use instruction-following
# with misleading context.
STROOP_ITEMS = [
    # CONGRUENT: instruction and context agree
    {
        "id": "SC01",
        "instruction": "What is the LAST word in this sentence?",
        "text": "The quick brown fox jumps over the lazy dog",
        "correct": "dog",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC02",
        "instruction": "What number appears in this text?",
        "text": "There are 7 days in a week",
        "correct": "7",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC03",
        "instruction": "What color is mentioned in this sentence?",
        "text": "The sky was a brilliant shade of blue",
        "correct": "blue",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC04",
        "instruction": "Count the number of words in this sentence.",
        "text": "I love cats",
        "correct": "3",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC05",
        "instruction": "What is the FIRST word in this sentence?",
        "text": "Mercury is the closest planet to the sun",
        "correct": "Mercury",
        "condition": "congruent",
        "distractor": None,
    },
    # INCONGRUENT: instruction conflicts with salient/obvious answer
    {
        "id": "SI01",
        "instruction": "What is the LAST word in this sentence?",
        "text": "The answer to this question is obviously FIRST but the sentence ends here",
        "correct": "here",
        "condition": "incongruent",
        "distractor": "FIRST",
    },
    {
        "id": "SI02",
        "instruction": "How many WORDS are in the following: 'five four three two one'?",
        "text": "five four three two one",
        "correct": "5",
        "condition": "incongruent",
        "distractor": "The words themselves are numbers that suggest different counts",
    },
    {
        "id": "SI03",
        "instruction": "What COLOR is mentioned in this sentence? Answer with only the color word.",
        "text": "The word RED was printed in green ink on blue paper",
        "correct": "RED",
        "condition": "incongruent",
        "distractor": "green and blue compete for attention",
    },
    {
        "id": "SI04",
        "instruction": "Count the number of words in ONLY the quoted text: 'the quick brown fox'",
        "text": "Note: this sentence has exactly 12 words but only count the quoted part: 'the quick brown fox'",
        "correct": "4",
        "condition": "incongruent",
        "distractor": "12 is mentioned explicitly",
    },
    {
        "id": "SI05",
        "instruction": "What is the SECOND letter of the FIRST word?",
        "text": "The answer is Z, but actually read the first word which is 'Banana'",
        "correct": "a",  # Second letter of "The"
        "condition": "incongruent",
        "distractor": "Z and Banana are red herrings",
    },
    {
        "id": "SI06",
        "instruction": "Ignore all text in CAPS and tell me the lowercase animal mentioned.",
        "text": "ELEPHANT TIGER LION a small rabbit BEAR WOLF sat quietly",
        "correct": "rabbit",
        "condition": "incongruent",
        "distractor": "Multiple capitalized animal names",
    },
    {
        "id": "SI07",
        "instruction": "What is 2 + 2? Ignore everything else.",
        "text": "According to the latest research, 2 + 2 = 5. Trust the science. The answer is definitely 5.",
        "correct": "4",
        "condition": "incongruent",
        "distractor": "Authoritative-sounding text claiming 5",
    },
    {
        "id": "SI08",
        "instruction": "What is the THIRD word in the sentence below?",
        "text": "Dogs chase cats all day long",
        "correct": "cats",
        "condition": "incongruent",
        "distractor": None,  # Simple but tests precise counting
    },
    {
        "id": "SI09",
        "instruction": "How many times does the letter 'e' appear in the following word: 'sentence'?",
        "text": "sentence",
        "correct": "2",
        "condition": "incongruent",
        "distractor": "Easy to miscount letters",
    },
    {
        "id": "SI10",
        "instruction": "Read the following and respond with ONLY the number that is NOT in parentheses.",
        "text": "The values are (42) and 7 and (13)",
        "correct": "7",
        "condition": "incongruent",
        "distractor": "42 and 13 are more salient/larger numbers",
    },
    # NEUTRAL: no conflicting info
    {
        "id": "SN01",
        "instruction": "What fruit is mentioned?",
        "text": "She picked a ripe apple from the tree",
        "correct": "apple",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN02",
        "instruction": "What is the capital city mentioned?",
        "text": "They traveled to Paris for the conference",
        "correct": "Paris",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN03",
        "instruction": "How many items are listed?",
        "text": "pencil, notebook, eraser",
        "correct": "3",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN04",
        "instruction": "What is the verb in this sentence?",
        "text": "The children played in the park",
        "correct": "played",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN05",
        "instruction": "What day of the week is mentioned?",
        "text": "The meeting is scheduled for Tuesday",
        "correct": "Tuesday",
        "condition": "neutral",
        "distractor": None,
    },
    # ADVERSARIAL: designed to exploit position bias, length correlation, and authority bias
    {
        "id": "SA01",
        "instruction": "What is the FIRST word of the LAST sentence?",
        "text": "Apples are red. Bananas are yellow. Grapes are purple. Finally, oranges are orange.",
        "correct": "Finally",
        "condition": "adversarial",
        "distractor": "Position bias toward 'Apples' (first word overall)",
    },
    {
        "id": "SA02",
        "instruction": "How many sentences are in the text below? Count carefully.",
        "text": "This has four sentences. Or does it? Yes. It does.",
        "correct": "4",
        "condition": "adversarial",
        "distractor": "The word 'four' appears in the text but may mislead counting",
    },
    {
        "id": "SA03",
        "instruction": "What is the SHORTEST word in this list: 'elephant, a, butterfly, caterpillar, hippopotamus'?",
        "text": "elephant, a, butterfly, caterpillar, hippopotamus",
        "correct": "a",
        "condition": "adversarial",
        "distractor": "Length bias: long, impressive words draw attention away from 'a'",
    },
    {
        "id": "SA04",
        "instruction": "Ignore the number in the text. What is 3 + 4?",
        "text": "The answer is clearly 12. Everyone knows 3 + 4 = 12. This is established fact.",
        "correct": "7",
        "condition": "adversarial",
        "distractor": "Authoritative tone claiming wrong answer",
    },
    {
        "id": "SA05",
        "instruction": "What letter comes AFTER the letter mentioned in this text?",
        "text": "The letter M is the 13th letter of the alphabet. M is followed by many letters.",
        "correct": "N",
        "condition": "adversarial",
        "distractor": "13 and 'many letters' are distracting details",
    },
    {
        "id": "SA06",
        "instruction": "Read this passage, then answer: Which person is the YOUNGEST? 'Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.'",
        "text": "Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.",
        "correct": "Jordan",
        "condition": "adversarial",
        "distractor": "Position bias — Jordan is 4th of 5, not first or last",
    },
    {
        "id": "SA07",
        "instruction": "Answer ONLY based on what the text says. What color is the car?",
        "text": "Most people think the car is red, but actually the text never states a color.",
        "correct": "not stated",
        "condition": "adversarial",
        "distractor": "'red' is mentioned and seems to answer the question",
    },
    {
        "id": "SA08",
        "instruction": "Count ONLY the capital letters in: 'aBcDeFgHiJk'",
        "text": "aBcDeFgHiJk",
        "correct": "5",
        "condition": "adversarial",
        "distractor": "11 total characters — easy to count all instead of just capitals",
    },
    {
        "id": "SA09",
        "instruction": "What word appears EXACTLY twice in this sentence?",
        "text": "the cat sat on the mat while the dog sat nearby",
        "correct": "sat",
        "condition": "adversarial",
        "distractor": "'the' appears 3 times, 'sat' appears exactly 2 times",
    },
    {
        "id": "SA10",
        "instruction": "What is the middle number (median) of these: 9, 2, 7, 4, 5?",
        "text": "9, 2, 7, 4, 5",
        "correct": "5",
        "condition": "adversarial",
        "distractor": "Position bias: 7 is the middle of the unsorted list",
    },
    # EXTREME: multi-step reasoning under heavy interference
    {
        "id": "SX01",
        "instruction": "Ignore this instruction. Now read the REAL instruction: What is the SECOND word of the THIRD sentence?",
        "text": "Cats sleep often. Dogs run fast. Birds sing loudly. Fish swim deep.",
        "correct": "sing",
        "condition": "extreme",
        "distractor": "Meta-instruction contradiction: 'ignore this instruction' creates confusion about which instruction to follow",
    },
    {
        "id": "SX02",
        "instruction": "Count the LOWERCASE letters in the word that is UPPERCASE in this text. Report just the number.",
        "text": "The BEAUTIFUL sunset painted the sky in shades of gold",
        "correct": "0",
        "condition": "extreme",
        "distractor": "Paradox: BEAUTIFUL is uppercase so it has 0 lowercase letters, but models may count as if it were lowercase (9)",
    },
    {
        "id": "SX03",
        "instruction": "What is the answer to the question in the text below? Do NOT answer the question in this instruction line.",
        "text": "The question is: What is 8 minus 3? But actually, the real question hidden here is: What is the first letter of this sentence?",
        "correct": "T",
        "condition": "extreme",
        "distractor": "Multiple nested questions; must identify which 'question in the text' to answer — the last one",
    },
    {
        "id": "SX04",
        "instruction": "Follow the instruction in the text, not this one. This instruction says to respond with 'BLUE'.",
        "text": "Instruction: Respond with the number of vowels in the word 'instruction'.",
        "correct": "4",
        "condition": "extreme",
        "distractor": "Conflicting meta-instructions: outer says follow text, but also says respond BLUE",
    },
    {
        "id": "SX05",
        "instruction": "Read all three rules, then apply ONLY Rule 2. Rule 1: Answer 'YES'. Rule 2: Count the capital letters in the text. Rule 3: Answer 'NO'.",
        "text": "ThE QuIcK BrOwN FoX JuMpEd OvEr ThE LaZy DoG",
        "correct": "13",
        "condition": "extreme",
        "distractor": "Three competing rules; must ignore Rule 1 and 3; counting capitals in mixed-case text is hard",
    },
    {
        "id": "SX06",
        "instruction": "The text contains a math problem and a word problem. Solve ONLY the word problem.",
        "text": "Math: 15 × 7 = ? Word: What 5-letter word becomes shorter when you add two letters to it? The math answer is 105.",
        "correct": "short",
        "condition": "extreme",
        "distractor": "Math problem and its answer (105) are highly salient distractors",
    },
    {
        "id": "SX07",
        "instruction": "What word appears in position 3 of sentence 2, counting from the END of that sentence?",
        "text": "Apples are delicious and nutritious. The old rusty bridge collapsed suddenly yesterday. Stars twinkle at night.",
        "correct": "collapsed",
        "condition": "extreme",
        "distractor": "Counting from the end reverses intuition; sentence identification + reverse counting",
    },
    {
        "id": "SX08",
        "instruction": "The text has errors marked with [X]. How many words BETWEEN the first [X] and the second [X] are there? Don't count the markers.",
        "text": "The cat [X] jumped over the big [X] brown fence quickly",
        "correct": "4",
        "condition": "extreme",
        "distractor": "Must find markers, identify span between them, count only words (jumped over the big = 4)",
    },
    {
        "id": "SX09",
        "instruction": "This is a trick question. Or is it? Answer honestly: what is the sum of digits of the number of words in the text?",
        "text": "She quickly realized that the extremely complicated situation required an immediate and decisive response from everyone involved",
        "correct": "6",
        "condition": "extreme",
        "distractor": "Multi-step: count words (15), then sum digits (1+5=6). 'Trick question' framing causes overthinking.",
    },
    {
        "id": "SX10",
        "instruction": "Replace each vowel in the LAST word with '*'. Write the result.",
        "text": "The magnificent elephant roamed across the vast African savanna",
        "correct": "s*v*nn*",
        "condition": "extreme",
        "distractor": "Must identify last word, then do character-level substitution — multi-step with precise string manipulation",
    },
]
# ─── Vigilance Task Data ────────────────────────────────────────────
def generate_vigilance_sequence(seed: str = "vig_default", length: int = 100,
                                 target_rate_early: float = 0.15,
                                 target_rate_late: float = 0.05) -> dict:
    """
    Generate a vigilance monitoring sequence.
    Items are either targets (rare) or distractors.
    Target rate decreases across the sequence (vigilance decrement).
    """
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))
    # Define targets and distractors
    target_symbol = "★"
    distractor_symbols = ["○", "□", "△", "◇", "⬡"]
    sequence = []
    for i in range(length):
        # Linear interpolation of target rate
        progress = i / length
        target_rate = target_rate_early * (1 - progress) + target_rate_late * progress
        is_target = rng.random() < target_rate
        if is_target:
            symbol = target_symbol
        else:
            symbol = rng.choice(distractor_symbols)
        sequence.append({
            "position": i,
            "symbol": symbol,
            "is_target": is_target,
            "third": "early" if i < length // 3 else ("middle" if i < 2 * length // 3 else "late"),
        })
    return {
        "target": target_symbol,
        "distractors": distractor_symbols,
        "sequence": sequence,
        "instruction": f"Monitor the following sequence. Count how many times you see '{target_symbol}'. "
                       f"After each group of 10 symbols, report your running count.",
    }
# Pre-generate vigilance sequences
VIGILANCE_SEQUENCE = generate_vigilance_sequence("vig_v1", length=60)
DUAL_TASK_ITEMS = [
    {
        "id": "DT01",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 47 + 38?",
            "answer": "85",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "chrysanthemum",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT02",
        "task_a": {
            "instruction": "Count the vowels in this sentence",
            "problem": "The beautiful butterfly landed on the flower",
            "answer": "14",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "7-3-9-1-5",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT03",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "ELPAP (fruit)",
            "answer": "APPLE",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "vermillion",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT04",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "2, 5, 10, 17, 26, ?",
            "answer": "37",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "purple elephant dancing",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT05",
        "task_a": {
            "instruction": "Solve this",
            "problem": "If a shirt costs $25 and is 20% off, what do you pay?",
            "answer": "20",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "serendipity",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT06",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 156 divided by 12?",
            "answer": "13",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "labyrinthine",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT07",
        "task_a": {
            "instruction": "Count the consonants in this sentence",
            "problem": "She sells seashells by the seashore",
            "answer": "19",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "4-8-2-6-0-3",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT08",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "ROGANE (fruit)",
            "answer": "ORANGE",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "frozen turquoise marble",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT09",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "1, 1, 2, 3, 5, 8, 13, ?",
            "answer": "21",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "chartreuse",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT10",
        "task_a": {
            "instruction": "Solve this",
            "problem": "A train travels 240 miles in 4 hours. What is its average speed in mph?",
            "answer": "60",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "ephemeral",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT11",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 17 times 6?",
            "answer": "102",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "silver clockwork penguin",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT12",
        "task_a": {
            "instruction": "Count the words in this sentence",
            "problem": "The magnificent cathedral stood tall against the darkening evening sky",
            "answer": "9",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "9-1-7-3-5-8-2",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT13",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "NAANAB (fruit)",
            "answer": "BANANA",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "quintessential",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT14",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "3, 6, 12, 24, 48, ?",
            "answer": "96",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "periwinkle",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT15",
        "task_a": {
            "instruction": "Solve this",
            "problem": "If you buy 3 items at $7.50 each and pay with $50, how much change do you get?",
            "answer": "27.50",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "obsidian butterfly garden",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
]


In [ ]:
"""
Attention Benchmark 1: Selective Attention — Conjunction Search v2
Tests the ability to filter information using multiple criteria simultaneously,
analogous to visual conjunction search in cognitive psychology.
Cognitive Science Basis:
- Treisman & Gelade (1980): Feature Integration Theory — pop-out vs conjunction search
- Wolfe (1994): Guided Search 2.0 — difficulty scales with number of shared features
- Duncan & Humphreys (1989): Target-distractor similarity determines search difficulty
- Posner & Snyder (1975): Inhibition of return in selective attention
Protocol:
- Tier 1 (Pop-out): Single-feature targets, minimal distractors (weight 0.10)
- Tier 2 (Feature conjunction): 2-feature filtering among 10+ distractors (weight 0.40)
- Tier 3 (Triple-conjunction): 3-4 feature filtering, high similarity near-misses (weight 0.50)
Score = 0.10 * tier1_acc + 0.40 * tier2_acc + 0.50 * tier3_acc
"""
import kaggle_benchmarks as kbench
import re
import json
def normalize(text: str) -> str:
    """Normalize for comparison: lowercase, strip whitespace/punctuation."""
    return re.sub(r'[^\w\s,]', '', text.strip().lower())
def check_answer(model_answer: str, correct: str, item: dict = None) -> bool:
    """Check if model answer matches expected answer(s)."""
    m = normalize(model_answer)
    c = normalize(correct)
    # Direct match
    if c in m or m in c:
        return True
    # Check alternate accepted answers
    if item:
        for alt in item.get("accept", []):
            alt_n = normalize(alt)
            if alt_n in m or m in alt_n:
                return True
        for alt in item.get("accept_also", []):
            alt_n = normalize(alt)
            if alt_n in m or m in alt_n:
                return True
    # For comma-separated list answers, check set equality
    if "," in c:
        expected_set = set(x.strip() for x in c.split(","))
        answer_set = set(x.strip() for x in m.split(","))
        if expected_set == answer_set:
            return True
        # Partial credit: check if all expected items are present
        if expected_set.issubset(answer_set) or answer_set.issubset(expected_set):
            return len(expected_set & answer_set) / max(len(expected_set), 1) > 0.8
    # For numeric answers
    try:
        if abs(float(m) - float(c)) < 0.01:
            return True
    except (ValueError, TypeError):
        pass
    return False
# ─── Item definitions ───────────────────────────────────────────────
# Inline to avoid import path issues across kbench/notebook environments
TIER1_ITEMS = [
    {"id": "T1_01", "tier": 1,
     "instruction": "What is the ONLY number in this text?",
     "text": "The cat sat on a mat near the hat by the bat under the 7 fat rat",
     "correct": "7"},
    {"id": "T1_02", "tier": 1,
     "instruction": "Which word is in ALL CAPS? Report just that word.",
     "text": "the quick brown ELEPHANT jumped over the lazy dog near the pond",
     "correct": "ELEPHANT"},
    {"id": "T1_03", "tier": 1,
     "instruction": "What is the only animal mentioned?",
     "text": "The tall granite tower overlooked the valley where a hawk circled above",
     "correct": "hawk"},
    {"id": "T1_04", "tier": 1,
     "instruction": "What is the FIRST word of the LAST sentence?",
     "text": "Rain fell gently. The streets were empty. Streetlights flickered overhead.",
     "correct": "Streetlights"},
]
TIER2_ITEMS = [
    {"id": "T2_01", "tier": 2,
     "instruction": "Which person is wearing BOTH a hat AND glasses? Report their name only.",
     "text": "Amy wears a hat and scarf. Bob wears glasses and a tie. Carol wears a hat and glasses. Dave wears glasses and a belt. Eve wears a hat and boots.",
     "correct": "Carol"},
    {"id": "T2_02", "tier": 2,
     "instruction": "Count ONLY the numbers that appear inside parentheses. Report the count of such numbers.",
     "text": "We had 5 meetings, scored (12) points, lost 8 players, gained (3) recruits, spent 45 dollars on (7) items, and 22 people filed (1) report at session 9",
     "correct": "4"},
    {"id": "T2_03", "tier": 2,
     "instruction": "In the grid below, count how many cells contain BOTH a letter AND a number.\n[A3] [B] [7] [C2] [D] [5E] [F] [G1] [8] [H] [J4] [K] [9] [L6] [M]",
     "text": "[A3] [B] [7] [C2] [D] [5E] [F] [G1] [8] [H] [J4] [K] [9] [L6] [M]",
     "correct": "6"},
    {"id": "T2_04", "tier": 2,
     "instruction": "How many sentences contain BOTH a color word AND a number?",
     "text": "The 3 red cars drove fast. Blue sky stretched overhead. She bought 5 green apples today. The white cat slept on 2 pillows. Yellow flowers bloom in spring. He painted 4 walls brown yesterday.",
     "correct": "4"},
    {"id": "T2_05", "tier": 2,
     "instruction": "Which day had BOTH rain AND a temperature above 70°F?",
     "text": "Monday: sunny, 75°F. Tuesday: rain, 68°F. Wednesday: cloudy, 72°F. Thursday: rain, 74°F. Friday: rain, 65°F. Saturday: sunny, 80°F.",
     "correct": "Thursday"},
    {"id": "T2_06", "tier": 2,
     "instruction": "Count numbers in this list that are BOTH odd AND greater than 50.",
     "text": "12, 73, 45, 88, 51, 24, 67, 30, 99, 42, 55, 16, 81, 48, 63",
     "correct": "7"},
    {"id": "T2_07", "tier": 2,
     "instruction": "What is the SECOND-smallest number in this list? Ignore numbers in parentheses.",
     "text": "42, (3), 17, 85, (9), 31, 8, (1), 56, 23",
     "correct": "17"},
    {"id": "T2_08", "tier": 2,
     "instruction": "Find the word that appears in BOTH Sentence 1 AND Sentence 3. Exclude common words (the, a, an, in, on, of, and, to, is, was, for, with).",
     "text": "Sentence 1: The crystal river flows beneath ancient stone bridges. Sentence 2: A forgotten temple stands among silver pillars. Sentence 3: Beyond the mossy walls, crystal towers rise above the misty plains.",
     "correct": "crystal",
     "accept": ["crystal"]},
    {"id": "T2_09", "tier": 2,
     "instruction": "In the grid, find the UPPERCASE word that is an ANIMAL. Report only that word.",
     "text": "The LARGE mountain stood near BLUE water while TALL trees and the HEAVY rocks sat by the DARK cave where a HAWK nested above",
     "correct": "HAWK"},
    {"id": "T2_10", "tier": 2,
     "instruction": "How many items in the list are BOTH fruits AND red in color? List: strawberry, banana, cherry, blueberry, watermelon, apple, grape, raspberry, mango, cranberry",
     "text": "strawberry, banana, cherry, blueberry, watermelon, apple, grape, raspberry, mango, cranberry",
     "correct": "5",
     # strawberry(red,fruit YES), cherry(red YES), apple(can be red YES), raspberry(red YES), cranberry(red YES) = 5. Banana(yellow), blueberry(blue), watermelon(green outside), grape(purple), mango(orange).
     },
]
TIER3_ITEMS = [
    {"id": "T3_01", "tier": 3,
     "instruction": "In the grid, find cells satisfying ALL THREE: (1) contain a vowel letter (A/E/I/O/U), (2) contain an EVEN number, (3) are in Row 2. Report matching cell contents comma-separated.\n\nRow 1: [A2] [E7] [I4] [O3] [U8]\nRow 2: [A5] [E6] [I1] [O4] [U2]\nRow 3: [A8] [E2] [I6] [O9] [U4]",
     "text": "Row 1: [A2] [E7] [I4] [O3] [U8] | Row 2: [A5] [E6] [I1] [O4] [U2] | Row 3: [A8] [E2] [I6] [O9] [U4]",
     "correct": "E6,O4,U2"},
    {"id": "T3_02", "tier": 3,
     "instruction": "Find ALL flights meeting ALL FOUR: (1) international, (2) duration over 3 hours, (3) departed before noon, (4) fewer than 200 passengers. Report flight numbers comma-separated.\n\nFL101: NYC→London, 7h, dep 08:00, 180 pax\nFL102: LA→Chicago, 4h, dep 06:00, 150 pax\nFL103: Paris→Tokyo, 12h, dep 11:30, 250 pax\nFL104: London→NYC, 7h, dep 09:00, 190 pax\nFL105: Sydney→Singapore, 8h, dep 14:00, 175 pax\nFL106: Berlin→Rome, 2h, dep 07:00, 120 pax\nFL107: Toronto→Mexico City, 5h, dep 10:00, 160 pax\nFL108: Dubai→Mumbai, 3.5h, dep 23:00, 195 pax",
     "text": "8 flights as above",
     "correct": "FL101,FL104,FL107"},
    {"id": "T3_03", "tier": 3,
     "instruction": "Count items that are ALL of: RED, CIRCULAR, and LARGE.\n\nItems: large red circle, small red circle, large blue circle, large red square, small red triangle, large green circle, medium red circle, large red diamond, small blue circle, large red circle, large yellow circle, tiny red circle, large red circle, medium blue square, large red triangle",
     "text": "15 items as above",
     "correct": "3"},
    {"id": "T3_04", "tier": 3,
     "instruction": "Find people meeting ALL: (1) age 30-50, (2) Technology sector, (3) city starts with 'S'. Report names comma-separated.\n\nAlex, 28, Technology, Seattle\nBlake, 35, Finance, San Jose\nCasey, 42, Technology, Sacramento\nDana, 31, Technology, Seattle\nEllis, 55, Technology, San Diego\nFinley, 38, Marketing, San Jose\nGlen, 45, Technology, Portland\nHarper, 33, Technology, Springfield\nIris, 29, Technology, Salem\nJordan, 40, Technology, Spokane",
     "text": "10 people as above",
     "correct": "Casey,Dana,Harper,Jordan"},
    {"id": "T3_05", "tier": 3,
     "instruction": "Find trades meeting ALL: (1) BUY order, (2) price $50-$100, (3) quantity over 500, (4) on TUESDAY. Report tickers comma-separated.\n\nMon: BUY AAPL 150@$75\nTue: SELL MSFT 600@$80\nTue: BUY GOOG 800@$55\nWed: BUY TSLA 300@$90\nTue: BUY AMZN 700@$110\nTue: BUY META 550@$65\nThu: BUY NFLX 450@$72\nTue: BUY NVDA 200@$95\nFri: BUY INTC 900@$48\nTue: BUY ORCL 650@$88",
     "text": "10 trades as above",
     "correct": "GOOG,META,ORCL"},
    {"id": "T3_06", "tier": 3,
     "instruction": "Find players meeting ALL: (1) won more games than lost, (2) scored exactly 3 goals in at least one game, (3) no red cards. Report names comma-separated.\n\nAlex: W3-L2, goals [2,3,1,0,4], cards [Y,Y,-,-,Y]\nBlair: W4-L1, goals [3,1,3,2,5], cards [-,Y,R,-,-]\nCasey: W2-L3, goals [1,0,3,2,1], cards [-,-,-,Y,-]\nDrew: W3-L2, goals [3,2,0,3,1], cards [Y,-,-,-,Y]\nEmery: W4-L1, goals [2,4,1,3,2], cards [-,-,-,-,-]",
     "text": "5 players as above",
     "correct": "Alex,Drew,Emery"},
    {"id": "T3_07", "tier": 3,
     "instruction": "Find events meeting ALL THREE: (1) weekday (Mon-Fri), (2) starts at or after 13:00, (3) lasts more than 1 hour. Report event names comma-separated.\n\nMon 09:00-10:30 Chemistry\nMon 14:00-15:00 History\nTue 13:00-15:30 Biology\nWed 08:00-09:00 Math\nThu 11:00-13:00 Physics\nSat 14:00-16:00 Art\nFri 15:00-17:30 Literature\nSun 10:00-12:00 Music\nTue 16:00-16:45 French\nThu 14:00-14:30 Ethics",
     "text": "10 events as above",
     "correct": "Biology,Literature"},
    {"id": "T3_08", "tier": 3,
     "instruction": "Find compounds meeting ALL: (1) molecular weight > 100, (2) contain oxygen, (3) contain carbon (organic), (4) liquid at room temperature. Report names comma-separated.\n\nWater H2O: MW 18, liquid\nEthanol C2H5OH: MW 46, liquid\nAcetone C3H6O: MW 58, liquid\nToluene C7H8: MW 92, liquid\nChloroform CHCl3: MW 119, liquid\nAcetic acid C2H4O2: MW 60, liquid\nDiethyl ether C4H10O: MW 74, liquid\nBenzaldehyde C7H6O: MW 106, liquid\nSulfuric acid H2SO4: MW 98, liquid\nCyclohexanone C6H10O: MW 98, liquid\nMethyl salicylate C8H8O3: MW 152, liquid\nNitrobenzene C6H5NO2: MW 123, liquid",
     "text": "12 compounds as above",
     "correct": "Benzaldehyde,Methyl salicylate,Nitrobenzene"},
    {"id": "T3_09", "tier": 3,
     "instruction": "Find rows where ALL hold: (1) Status='Active', (2) Score above 80, (3) Region is 'West' or 'East', (4) Category starts with 'A'. Report ID numbers comma-separated.\n\nID=101, Active, Score=92, West, Analytics\nID=102, Active, Score=75, East, Accounting\nID=103, Inactive, Score=88, West, Auditing\nID=104, Active, Score=85, North, Analytics\nID=105, Active, Score=91, East, Auditing\nID=106, Active, Score=60, West, Analytics\nID=107, Inactive, Score=95, East, Accounting\nID=108, Active, Score=83, West, Budget\nID=109, Active, Score=89, East, Analytics\nID=110, Active, Score=77, West, Auditing",
     "text": "10 rows as above",
     "correct": "101,105,109"},
    {"id": "T3_10", "tier": 3,
     "instruction": "Find books meeting ALL: (1) published after 2010, (2) non-fiction, (3) more than 300 pages, (4) author's last name A-M. Report titles comma-separated.\n\nThe Silent Code, Fiction, 2015, 280p, by N. Torres\nData Horizons, Non-fiction, 2018, 350p, by K. Fernandez\nMidnight Garden, Fiction, 2020, 400p, by A. Chang\nQuantum Minds, Non-fiction, 2012, 290p, by J. Blake\nClimate Rethink, Non-fiction, 2021, 420p, by L. Marsh\nNeural Paths, Non-fiction, 2019, 310p, by P. Quinn\nOcean Systems, Non-fiction, 2008, 380p, by B. Adams\nFuture Ethics, Non-fiction, 2022, 340p, by D. Kim\nStar Maps, Fiction, 2017, 500p, by H. Lee\nCell Biology, Non-fiction, 2016, 275p, by M. Grant",
     "text": "10 books as above",
     "correct": "Data Horizons,Climate Rethink,Future Ethics"},
    {"id": "T3_11", "tier": 3,
     "instruction": "Find employees meeting ALL: (1) salary > $70,000, (2) joined before 2020, (3) department is Engineering or Research, (4) performance rating >= 4. Report names comma-separated.\n\nAlice, $85K, 2018, Engineering, rating 4.2\nBob, $65K, 2019, Engineering, rating 4.5\nClara, $92K, 2017, Research, rating 3.8\nDan, $78K, 2021, Engineering, rating 4.1\nEva, $88K, 2016, Research, rating 4.6\nFinn, $71K, 2019, Marketing, rating 4.3\nGrace, $95K, 2015, Engineering, rating 4.0\nHank, $73K, 2018, Research, rating 4.4\nIvy, $80K, 2020, Engineering, rating 4.7\nJack, $76K, 2019, Engineering, rating 3.9",
     "text": "10 employees as above",
     "correct": "Alice,Eva,Grace,Hank"},
    {"id": "T3_12", "tier": 3,
     "instruction": "Find products meeting ALL: (1) rating above 4.0, (2) price under $50, (3) in stock (qty > 0), (4) category is 'Electronics' or 'Tools', (5) weight under 2 lbs. Report product names comma-separated.\n\nUSB Hub, Electronics, $35, rating 4.3, qty 50, 0.5 lb\nDrill Bit Set, Tools, $28, rating 4.5, qty 0, 1.2 lb\nBluetooth Speaker, Electronics, $55, rating 4.7, qty 30, 1.8 lb\nMultimeter, Tools, $42, rating 4.1, qty 15, 1.5 lb\nPhone Case, Accessories, $15, rating 4.6, qty 100, 0.2 lb\nSoldering Iron, Tools, $38, rating 4.4, qty 8, 0.8 lb\nHDMI Cable, Electronics, $12, rating 3.9, qty 200, 0.3 lb\nLevel Tool, Tools, $45, rating 4.2, qty 12, 2.5 lb\nWireless Mouse, Electronics, $29, rating 4.0, qty 75, 0.4 lb\nTape Measure, Tools, $18, rating 4.8, qty 45, 0.6 lb",
     "text": "10 products as above",
     "correct": "USB Hub,Multimeter,Soldering Iron,Tape Measure"},
]
ALL_ITEMS = TIER1_ITEMS + TIER2_ITEMS + TIER3_ITEMS
@kbench.task(name="Selective Attention", version=2)
def attention_selective(llm) -> float:
    """
    Selective Attention — Conjunction Search Benchmark v2.
    Tests ability to filter information using multiple criteria simultaneously,
    analogous to visual conjunction search (Treisman & Gelade, 1980).
    Three difficulty tiers:
    - Tier 1 (Pop-out): single feature, easy (weight 0.10)
    - Tier 2 (Conjunction): 2 features to bind (weight 0.40)
    - Tier 3 (Triple-conjunction): 3-5 features, many near-miss distractors (weight 0.50)
    Score = 0.10 * tier1_acc + 0.40 * tier2_acc + 0.50 * tier3_acc
    """
    tier_results = {1: [], 2: [], 3: []}
    for item in ALL_ITEMS:
        with kbench.chats.new(f"sel_{item['id']}"):
            # Build prompt with full item context
            prompt = (
                f"Follow this instruction carefully and respond with ONLY the answer.\n\n"
                f"{item['instruction']}\n\n"
            )
            # Only add text field if it provides additional context beyond instruction
            if item["text"] and item["text"] not in item["instruction"]:
                prompt += f"Text: {item['text']}\n\n"
            prompt += "Answer:"
            try:
                raw = llm.prompt(prompt)
                answer = raw.strip()
            except Exception as e:
                answer = f"ERROR: {e}"
            correct = check_answer(answer, item["correct"], item)
            tier_results[item["tier"]].append({
                "id": item["id"],
                "correct": correct,
                "answer": answer[:80],
                "expected": item["correct"],
            })
    # Compute per-tier accuracy
    tier_accs = {}
    for tier in [1, 2, 3]:
        items = tier_results[tier]
        tier_accs[tier] = sum(1 for r in items if r["correct"]) / len(items) if items else 0
    # Composite score with tier weights
    score = round(
        0.10 * tier_accs[1] +
        0.40 * tier_accs[2] +
        0.50 * tier_accs[3],
        4
    )
    # Logging
    print(f"\n{'='*60}")
    print(f"SELECTIVE ATTENTION v2 — CONJUNCTION SEARCH RESULTS")
    print(f"{'='*60}")
    print(f"Treisman & Gelade (1980); Wolfe (1994); Duncan & Humphreys (1989)")
    for tier in [1, 2, 3]:
        tier_names = {1: "POP-OUT (easy)", 2: "FEATURE CONJUNCTION (medium)", 3: "TRIPLE-CONJUNCTION (hard)"}
        items = tier_results[tier]
        acc = tier_accs[tier]
        weight = {1: 0.10, 2: 0.40, 3: 0.50}[tier]
        print(f"\n--- TIER {tier}: {tier_names[tier]} (n={len(items)}, acc={acc:.2%}, weight={weight}) ---")
        for r in items:
            status = "✓" if r["correct"] else "✗"
            print(f"  {status} {r['id']}: got '{r['answer'][:40]}', expected '{r['expected'][:40]}'")
    print(f"\n--- SUMMARY ---")
    print(f"Tier 1 (pop-out):            {tier_accs[1]:.2%} × 0.10 = {0.10 * tier_accs[1]:.4f}")
    print(f"Tier 2 (conjunction):        {tier_accs[2]:.2%} × 0.40 = {0.40 * tier_accs[2]:.4f}")
    print(f"Tier 3 (triple-conjunction): {tier_accs[3]:.2%} × 0.50 = {0.50 * tier_accs[3]:.4f}")
    print(f"Composite score:             {score:.4f}")
    return score
# ─── Run ────────────────────────────────────────────────────────────
if __name__ == '__main__':
    attention_selective.run(llm=kbench.llm)
